In [ ]:
!pip -q install --upgrade "diffusers>=0.30.0" "transformers>=4.40.0" accelerate peft safetensors \
  torch torchvision torchaudio pandas numpy pillow tqdm scipy scikit-learn matplotlib


!pip -q install bitsandbytes || true
!pip -q install xformers || true

!pip -q install lpips torchxrayvision


In [ ]:
import os, random, math, json
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import matplotlib.pyplot as plt

from diffusers import StableDiffusionPipeline, DDPMScheduler
from diffusers.models import AutoencoderKL
from peft import LoraConfig, get_peft_model, PeftModel

import lpips
import torchxrayvision as xrv

CFG = dict(
    # Paths
    data_root=os.path.expanduser("~/data"),
    local_splits_subdir="xray/images_png",
    local_images_subdir="xray/images_png/images",

    train_csv_name="train_test_split_train_df.csv",
    val_csv_name="train_test_split_val_df.csv",

    img_col="Image Index",
    label_col="Finding Labels",

    # Selection
    per_label_train=500,
    per_label_val=250,
    include_no_finding=True,
    max_total_train=12000,
    max_total_val=2000,

    # Training 
    seed=42,
    base_model="runwayml/stable-diffusion-v1-5",
    img_size=512,

    batch_size=1,
    grad_accum=4,              
    epochs=6,                  
    max_train_steps=4500,
    lr=5e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    mixed_precision=True,
    num_workers=4,

    # LoRA
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.05,

    # Optional better VAE
    use_better_vae=True,

    # Eval / sampling
    eval_prompts_n=16,
    eval_seeds=[0, 1],
    num_inference_steps=30,
    guidance_scale=4.5,

    #  eval toggles
    run_lpips=True,
    run_xrv_eval=True,
)

def set_all_seeds(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_all_seeds(CFG["seed"])

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

DATA_ROOT = CFG["data_root"]
LOCAL_SPLITS_DIR = os.path.join(DATA_ROOT, CFG["local_splits_subdir"])
LOCAL_IMG_DIR    = os.path.join(DATA_ROOT, CFG["local_images_subdir"])
OUT_DIR          = os.path.join(DATA_ROOT, "outputs_paperworthy_lora")
os.makedirs(OUT_DIR, exist_ok=True)

print("LOCAL_SPLITS_DIR:", LOCAL_SPLITS_DIR)
print("LOCAL_IMG_DIR:", LOCAL_IMG_DIR)
print("OUT_DIR:", OUT_DIR)


In [ ]:
train_csv = os.path.join(LOCAL_SPLITS_DIR, CFG["train_csv_name"])
val_csv   = os.path.join(LOCAL_SPLITS_DIR, CFG["val_csv_name"])

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)

train_csv = os.path.join(LOCAL_SPLITS_DIR, CFG["train_csv_name"])
val_csv   = os.path.join(LOCAL_SPLITS_DIR, CFG["val_csv_name"])

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)

from sklearn.model_selection import train_test_split

IMG_COL = CFG["img_col"]
LBL_COL = CFG["label_col"]


def parse_labels(s: str):
    s = str(s).strip()
    if s.lower() == "no finding":
        return ["No Finding"]
    return [x.strip() for x in s.split("|") if x.strip()]


FINDING_PHRASES = {
    "Infiltration": "patchy increased lung opacities consistent with infiltration",
    "Effusion": "pleural effusion with blunting of the costophrenic angle",
    "Atelectasis": "linear atelectasis with mild volume loss",
    "Nodule": "solitary pulmonary nodule as a small round opacity",
    "Mass": "pulmonary mass as a larger rounded opacity",
    "Consolidation": "focal airspace consolidation with increased opacity",
    "Pleural_Thickening": "pleural thickening along the lateral chest wall",
    "Pneumothorax": "pneumothorax with a visible pleural line and reduced peripheral lung markings",
    "Cardiomegaly": "enlarged cardiac silhouette consistent with cardiomegaly",
    "Emphysema": "hyperinflated lungs with emphysematous changes",
    "Edema": "interstitial pulmonary edema with increased interstitial markings",
    "Fibrosis": "fibrotic changes with reticular opacities",
    "Pneumonia": "airspace opacities consistent with pneumonia",
    "Hernia": "hiatal hernia appearance",
}

def labels_to_prompt(label_str: str) -> str:
    label_str = str(label_str).strip()

    
    style = (
        "high-resolution frontal PA chest X-ray radiograph, grayscale, clinical, realistic, "
        "correct anatomy, sharp lung markings, no text, no letters, no symbols"
    )

    labs = [x.strip() for x in label_str.split("|") if x.strip()]
    labs = [x for x in labs if x.lower() != "no finding"]

    if len(labs) == 0:
        return style + ", normal study, no acute cardiopulmonary abnormality"

    phrases = [FINDING_PHRASES.get(lab, f"evidence of {lab.lower()}") for lab in labs]
    return style + ", abnormal findings: " + "; ".join(phrases)



def finalize(df_):
    df_ = df_[[IMG_COL, LBL_COL]].dropna(subset=[LBL_COL]).copy()
    df_["filepath"] = df_[IMG_COL].apply(lambda x: os.path.join(LOCAL_IMG_DIR, str(x)))
    df_["labels"] = df_[LBL_COL].apply(parse_labels)
    df_["label_name"] = df_["labels"].apply(lambda xs: xs[0] if isinstance(xs, list) and len(xs) else "No Finding")
    df_["prompt"] = df_[LBL_COL].apply(labels_to_prompt)
    return df_.reset_index(drop=True)

train_csv = os.path.join(LOCAL_SPLITS_DIR, "train.csv")
val_csv   = os.path.join(LOCAL_SPLITS_DIR, "val.csv")

if os.path.exists(train_csv) and os.path.exists(val_csv):
    train_df = finalize(pd.read_csv(train_csv))
    val_df   = finalize(pd.read_csv(val_csv))
    print("Using provided train/val CSVs.")
else:
    entry = pd.read_csv(os.path.join(LOCAL_SPLITS_DIR, "Data_Entry_2017_v2020.csv"))
    entry = entry[[IMG_COL, LBL_COL]].dropna().reset_index(drop=True)

    with open(os.path.join(LOCAL_SPLITS_DIR, "train_val_list.txt")) as f:
        trainval_files = [x.strip() for x in f if x.strip()]

    tv_df = entry[entry[IMG_COL].isin(trainval_files)].copy()
    tv_df = finalize(tv_df)

    train_df, val_df = train_test_split(tv_df, test_size=0.1, random_state=CFG["seed"], shuffle=True)
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    print("Built train/val from train_val_list.txt (10% val).")

print("raw train:", len(train_df), "raw val:", len(val_df))


In [ ]:
def select_union_per_label(df, per_label, include_no_finding=True, max_total=None, seed=42):
    rng = np.random.RandomState(seed)
    label_to_idxs = defaultdict(list)
    for i, labs in enumerate(df["labels"]):
        for l in labs:
            if (l == "No Finding") and (not include_no_finding):
                continue
            label_to_idxs[l].append(i)

    selected = set()
    for lbl, idxs in label_to_idxs.items():
        if len(idxs) == 0:
            continue
        k = min(per_label, len(idxs))
        pick = rng.choice(idxs, size=k, replace=False)
        selected.update(pick.tolist())

    selected = list(selected)
    rng.shuffle(selected)
    if max_total is not None and len(selected) > max_total:
        selected = selected[:max_total]

    return df.iloc[selected].reset_index(drop=True)

train_sel = select_union_per_label(
    train_df, per_label=CFG["per_label_train"],
    include_no_finding=CFG["include_no_finding"],
    max_total=CFG["max_total_train"], seed=CFG["seed"]
)
val_sel = select_union_per_label(
    val_df, per_label=CFG["per_label_val"],
    include_no_finding=True,
    max_total=CFG["max_total_val"], seed=CFG["seed"] + 1
)

print("selected train:", len(train_sel), "selected val:", len(val_sel))

# Save selections 
train_sel.to_csv(os.path.join(OUT_DIR, "train_selected.csv"), index=False)
val_sel.to_csv(os.path.join(OUT_DIR, "val_selected.csv"), index=False)


In [ ]:
def label_counts(df):
    c = Counter()
    for labs in df["labels"]:
        c.update(labs)
    return c

lc = label_counts(train_sel)
eda = pd.DataFrame(lc.items(), columns=["label","count"]).sort_values("count", ascending=False)
eda.to_csv(os.path.join(OUT_DIR, "label_counts_train_selected.csv"), index=False)
display(eda.head(20))

# plot label distribution
top = eda.head(15)
plt.figure(figsize=(10,4))
plt.bar(top["label"], top["count"])
plt.xticks(rotation=45, ha="right")
plt.title("Train (selected) label counts")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "eda_train_label_counts.png"), dpi=200)
plt.show()

# co-occurrence
from itertools import combinations
co = defaultdict(int)
for labs in train_sel["labels"]:
    labs = sorted(list(set(labs)))
    for a, b in combinations(labs, 2):
        co[(a,b)] += 1
co_df = pd.DataFrame([(a,b,c) for (a,b),c in co.items()],
                     columns=["label_a","label_b","count"]).sort_values("count", ascending=False)
co_df.to_csv(os.path.join(OUT_DIR, "cooccurrence_top.csv"), index=False)
display(co_df.head(20))

# sample image grid
def save_eda_samples(df, n=12, seed=42):
    samp = df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)
    cols = 4
    rows = int(np.ceil(len(samp)/cols))
    plt.figure(figsize=(4*cols, 4*rows))
    for i in range(len(samp)):
        img = Image.open(samp.loc[i, "filepath"]).convert("L")
        plt.subplot(rows, cols, i+1)
        plt.imshow(img, cmap="gray")
        plt.axis("off")
        plt.title(samp.loc[i, "prompt"][:55], fontsize=8)
    plt.tight_layout()
    outp = os.path.join(OUT_DIR, "eda_samples.png")
    plt.savefig(outp, dpi=200)
    plt.show()
    print("Saved:", outp)

save_eda_samples(train_sel, n=12, seed=CFG["seed"])


In [ ]:
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize(CFG["img_size"], interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(CFG["img_size"]),
    transforms.RandomApply([
        transforms.RandomAffine(
            degrees=4,
            translate=(0.03, 0.03),
            scale=(0.98, 1.02),
            interpolation=InterpolationMode.BILINEAR,
            fill=0
        )
    ], p=0.6),
    transforms.ToTensor(),                          
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),  
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize(CFG["img_size"], interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(CFG["img_size"]),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

class XrayPromptDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("L")  # TRUE grayscale
        img = self.transform(img)
        return {"pixel_values": img, "prompt": row["prompt"], "label_str": row[LBL_COL]}

train_ds = XrayPromptDataset(train_sel, train_tf)
val_ds   = XrayPromptDataset(val_sel, val_tf)


In [ ]:
# Inverse-frequency weights for multi-label examples
lc = label_counts(train_sel)
weights = []
for labs in train_sel["labels"]:
    inv = [1.0 / lc[l] for l in labs]
    weights.append(float(np.mean(inv)))
weights = torch.DoubleTensor(weights)

sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    sampler=sampler,
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)

b = next(iter(train_loader))
print("batch:", b["pixel_values"].shape, "prompt:", b["prompt"][0][:80])


In [ ]:
dtype = torch.float16 if (device=="cuda" and CFG["mixed_precision"]) else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(
    CFG["base_model"],
    torch_dtype=dtype,
    safety_checker=None,
).to(device)




pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.unet.enable_gradient_checkpointing()
try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xFormers enabled")
except Exception as e:
    print("xFormers not enabled:", e)

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
vae = pipe.vae
unet = pipe.unet

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

lora_config = LoraConfig(
    r=CFG["lora_r"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    bias="none",
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
)

unet_lora = get_peft_model(unet, lora_config).to(device)
unet_lora.train()
unet_lora.print_trainable_parameters()

# TRAINING noise scheduler 
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

# Optimizer 
try:
    import bitsandbytes as bnb
    optimizer = bnb.optim.AdamW8bit(unet_lora.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    print("Using AdamW8bit")
except Exception:
    optimizer = torch.optim.AdamW(unet_lora.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    print("Using AdamW")

scaler = torch.amp.GradScaler("cuda", enabled=(device=="cuda" and CFG["mixed_precision"]))

VAE_SCALE = 0.18215


In [ ]:
NEG_PROMPT = "color, rgb, blue, orange, chromatic aberration, cartoon, illustration, painting, CGI, stylized, text, watermark, label, symbols, frame, overexposed, halos, blurry"

def encode_prompts(prompts):
    tok = tokenizer(prompts, padding="max_length", truncation=True,
                    max_length=tokenizer.model_max_length, return_tensors="pt")
    with torch.no_grad():
        return text_encoder(tok.input_ids.to(device))[0]

@torch.no_grad()
def vae_encode(pixel_values):
    latents = vae.encode(pixel_values).latent_dist.sample()
    return latents * VAE_SCALE

# Main Training Loop (Step-Based) 

# Configuration
MAX_TRAIN_STEPS = CFG["max_train_steps"]  # Total optimization steps
VALIDATE_EVERY_STEPS = 500                # Run validation every 500 steps 
history = []
best_val = float("inf")
best_dir = os.path.join(OUT_DIR, "best_lora")
last_dir = os.path.join(OUT_DIR, "last_lora")

unet_lora.train()
optimizer.zero_grad(set_to_none=True)

global_step = 0      # Counts batches seen
optimizer_step = 0   # Counts weight updates
accum = 0            # Gradient accumulation counter
losses = []          # Running loss buffer

# Infinite iterator over the dataloader
train_iter = iter(train_loader)

progress_bar = tqdm(range(MAX_TRAIN_STEPS), desc="Training Steps")

while optimizer_step < MAX_TRAIN_STEPS:
    
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    # Forward Pass
    px = batch["pixel_values"].to(device, non_blocking=True)
    prompts = batch["prompt"]

    with torch.amp.autocast("cuda", enabled=(device=="cuda" and CFG["mixed_precision"])):
        lat = vae_encode(px)
        noise = torch.randn_like(lat)

        # optional noise offset 
        noise_off = 0.05
        if noise_off > 0:
            noise = noise + noise_off * torch.randn(lat.shape[0], lat.shape[1], 1, 1, device=device)

        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (lat.shape[0],), device=device).long()
        nlat = noise_scheduler.add_noise(lat, noise, t)
        enc = encode_prompts(prompts)
        pred = unet_lora(nlat, t, encoder_hidden_states=enc).sample

        # Loss calculation
        loss = F.mse_loss(pred.float(), noise.float()) / CFG["grad_accum"]

    # Backward Pass
    scaler.scale(loss).backward()
    accum += 1
    losses.append(loss.item() * CFG["grad_accum"])
    global_step += 1

    # Optimizer Step 
    if accum >= CFG["grad_accum"]:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(unet_lora.parameters(), CFG["max_grad_norm"])
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        accum = 0
        optimizer_step += 1
        progress_bar.update(1)

        # Update progress bar description
        avg_loss = np.mean(losses[-20:]) if losses else 0.0
        progress_bar.set_postfix(loss=f"{avg_loss:.4f}", step=optimizer_step)

        # VALIDATION 
        if optimizer_step % VALIDATE_EVERY_STEPS == 0:
            print(f"\n[Step {optimizer_step}] Running Validation...")

            # Validation Loop
            unet_lora.eval()
            val_losses = []
            for val_batch in tqdm(val_loader, desc="Validating", leave=False):
                with torch.no_grad():
                    px_val = val_batch["pixel_values"].to(device, non_blocking=True)
                    prompts_val = val_batch["prompt"]
                    with torch.amp.autocast("cuda", enabled=(device=="cuda" and CFG["mixed_precision"])):
                        lat_val = vae_encode(px_val)
                        noise_val = torch.randn_like(lat_val)
                        t_val = torch.randint(0, noise_scheduler.config.num_train_timesteps, (lat_val.shape[0],), device=device).long()
                        nlat_val = noise_scheduler.add_noise(lat_val, noise_val, t_val)
                        enc_val = encode_prompts(prompts_val)
                        pred_val = unet_lora(nlat_val, t_val, encoder_hidden_states=enc_val).sample
                        loss_val = F.mse_loss(pred_val.float(), noise_val.float())
                        val_losses.append(loss_val.item())

            current_val_loss = float(np.mean(val_losses))
            print(f"[Step {optimizer_step}] Train Loss: {avg_loss:.4f} | Val Loss: {current_val_loss:.4f}")

            # Save History
            history.append({
                "step": optimizer_step,
                "train_loss": avg_loss,
                "val_loss": current_val_loss
            })
            pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "loss_history.csv"), index=False)

            # Save Best Model
            if current_val_loss < best_val:
                best_val = current_val_loss
                os.makedirs(best_dir, exist_ok=True)
                unet_lora.save_pretrained(best_dir)
                with open(os.path.join(best_dir, "cfg.json"), "w") as f:
                    json.dump(CFG, f, indent=2)
                print(f" -> New Best Model Saved! (Val Loss: {best_val:.4f})")

            # Switch back to train mode
            unet_lora.train()

# Save final model at the very end
os.makedirs(last_dir, exist_ok=True)
unet_lora.save_pretrained(last_dir)
print(f"Training Complete. Last model saved to {last_dir}")
print(f"Best model saved to {best_dir}")

# Plotting the Step-based history
hist = pd.DataFrame(history)
if not hist.empty:
    plt.figure(figsize=(6,4))
    plt.plot(hist["step"], hist["train_loss"], label="train")
    plt.plot(hist["step"], hist["val_loss"], label="val")
    plt.xlabel("Optimizer Steps"); plt.ylabel("MSE Loss")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "loss_curve.png"), dpi=200)
    plt.show()

In [ ]:
def load_pipe_base():
    p = StableDiffusionPipeline.from_pretrained(
        CFG["base_model"],
        torch_dtype=dtype,
        safety_checker=None,
    ).to(device)





    p.enable_attention_slicing()
    p.enable_vae_slicing()
    try:
        p.enable_xformers_memory_efficient_attention()
    except Exception:
        pass
    p.set_progress_bar_config(disable=True)
    return p

def load_pipe_with_lora(adapter_dir):
    p = load_pipe_base()
    peft_unet = PeftModel.from_pretrained(p.unet, adapter_dir)
    p.unet = peft_unet.merge_and_unload()
    p.set_progress_bar_config(disable=True)
    return p

base_pipe = load_pipe_base()
lora_pipe = load_pipe_with_lora(best_dir)

@torch.inference_mode()
def gen_one(pipe_, prompt, seed):
    g = torch.Generator(device=device).manual_seed(int(seed))
    return pipe_(
        prompt=prompt,
        negative_prompt=NEG_PROMPT,
        num_inference_steps=CFG["num_inference_steps"],
        guidance_scale=CFG["guidance_scale"],
        generator=g
    ).images[0]

# deterministic eval subset
eval_df = val_sel.sample(n=min(len(val_sel), CFG["eval_prompts_n"]), random_state=CFG["seed"]).reset_index(drop=True)
eval_prompts = eval_df["prompt"].tolist()
eval_labels  = eval_df[LBL_COL].tolist()

# save prompts for reproducibility
eval_prompts_path = os.path.join(OUT_DIR, "eval_prompts.csv")
pd.DataFrame({"prompt_id": range(len(eval_prompts)), "prompt": eval_prompts, "label_str": eval_labels}).to_csv(eval_prompts_path, index=False)
print("Saved eval prompts ->", eval_prompts_path)

base_dir = os.path.join(OUT_DIR, "gen", "base")
lora_dir = os.path.join(OUT_DIR, "gen", "lora")
os.makedirs(base_dir, exist_ok=True)
os.makedirs(lora_dir, exist_ok=True)

seed0 = CFG["eval_seeds"][0]
pairs = []
records = []

for i, (prompt, label) in enumerate(tqdm(list(zip(eval_prompts, eval_labels)), desc="base vs lora")):
    img_base = gen_one(base_pipe, prompt, seed0)
    img_lora = gen_one(lora_pipe, prompt, seed0)

    base_path = os.path.join(base_dir, f"{i}.png")
    lora_path = os.path.join(lora_dir, f"{i}.png")
    img_base.save(base_path)
    img_lora.save(lora_path)

    pairs.append([img_base, img_lora])
    records.append({
        "prompt_id": i,
        "prompt": prompt,
        "label_str": label,
        "seed": seed0,
        "base_image_path": base_path,
        "lora_image_path": lora_path
    })

comparison_csv = os.path.join(OUT_DIR, "base_vs_lora_comparison.csv")
pd.DataFrame(records).to_csv(comparison_csv, index=False)
print("Saved comparison metadata ->", comparison_csv)

# grid image 
w, h = pairs[0][0].size
grid = Image.new("RGB", (2*w, len(pairs)*h))
for r in range(len(pairs)):
    grid.paste(pairs[r][0], (0, r*h))
    grid.paste(pairs[r][1], (w, r*h))

grid_path = os.path.join(OUT_DIR, "base_vs_lora_grid.png")
grid.save(grid_path)
display(grid)
print("Saved grid ->", grid_path)


In [ ]:
import os
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import textwrap

def _load_font(size=20):
    try:
        return ImageFont.truetype("DejaVuSans.ttf", size)
    except:
        return ImageFont.load_default()

def make_comparison_sheet_gray(
    examples_df,
    out_path,
    label_col="label_name",
    prompt_col="prompt",
    base_img_col="base_path",
    lora_img_col="lora_path",
    base_txt_col="base_text",
    lora_txt_col="lora_text",
    n=15,
    img_size=256,
    pad=18,
):
    df = examples_df.head(n).reset_index(drop=True)

    font_title   = _load_font(22)
    font_small   = _load_font(16)
    font_caption = _load_font(15)

    # Grayscale canvas
    W = pad + img_size + pad + img_size + pad
    header_base_h = 70
    colhdr_h = 30
    caption_h = 60
    row_gap = 10

    tmp_draw = ImageDraw.Draw(Image.new("L", (10, 10), 255))
    header_heights = []
    wrapped_headers = []

    for i in range(len(df)):
        row = df.loc[i]
        label = str(row.get(label_col, ""))
        prompt = str(row.get(prompt_col, ""))

        header_text = f"Label: {label}\nPrompt: {prompt}"
        wrapped = "\n".join([textwrap.fill(line, width=70) for line in header_text.split("\n")])

        bbox = tmp_draw.multiline_textbbox((0, 0), wrapped, font=font_small, spacing=6)
        text_h = (bbox[3] - bbox[1])
        header_h = max(header_base_h, text_h + 10)

        wrapped_headers.append(wrapped)
        header_heights.append(header_h)

    row_heights = [header_heights[i] + colhdr_h + img_size + caption_h + row_gap for i in range(len(df))]
    H = pad + sum(row_heights) + pad

    canvas = Image.new("L", (W, H), 255)  # white background in grayscale
    draw = ImageDraw.Draw(canvas)

    def _safe_open_L(path):
        try:
            return Image.open(path).convert("L")
        except:
            return Image.new("L", (img_size, img_size), 255)

    y = pad
    for i in range(len(df)):
        row = df.loc[i]
        wrapped = wrapped_headers[i]
        header_h = header_heights[i]

        # Header centered
        bbox = draw.multiline_textbbox((0, 0), wrapped, font=font_small, spacing=6)
        text_w = bbox[2] - bbox[0]
        draw.multiline_text(((W - text_w) / 2, y), wrapped, font=font_small, fill=0, spacing=6)

        y_img_top = y + header_h
        left_x  = pad
        right_x = pad + img_size + pad

        # Column titles
        draw.text((left_x,  y_img_top), "Base model", font=font_title, fill=0)
        draw.text((right_x, y_img_top), "LoRA model", font=font_title, fill=0)

        y_imgs = y_img_top + colhdr_h

        base_im = _safe_open_L(row.get(base_img_col, ""))
        lora_im = _safe_open_L(row.get(lora_img_col, ""))

        base_im = base_im.resize((img_size, img_size))
        lora_im = lora_im.resize((img_size, img_size))

        canvas.paste(base_im, (left_x,  y_imgs))
        canvas.paste(lora_im, (right_x, y_imgs))

        # Captions
        y_cap = y_imgs + img_size + 8
        base_cap = str(row.get(base_txt_col, "")) if base_txt_col in df.columns else ""
        lora_cap = str(row.get(lora_txt_col, "")) if lora_txt_col in df.columns else ""

        base_cap = textwrap.fill(base_cap, width=32)
        lora_cap = textwrap.fill(lora_cap, width=32)

        draw.multiline_text((left_x,  y_cap), base_cap, font=font_caption, fill=0, spacing=4)
        draw.multiline_text((right_x, y_cap), lora_cap, font=font_caption, fill=0, spacing=4)

        y += row_heights[i]

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    canvas.save(out_path)
    return out_path


In [ ]:
import os
import pandas as pd
import torch


# Settings

N_SAMPLES = 15

# save output 
VIZ_SAVE_DIR = os.path.join(OUT_DIR, "viz_outputs_gray")
SHEET_PATH   = os.path.join(OUT_DIR, "sheet_gray.png")

BASE_SEED = 1000
GUIDANCE = 10.0
STEPS = 35
IMG_SIZE = 512

NEG_PROMPT = "normal lungs, no abnormality, clear lungs, perfectly normal, text, letters, watermark, symbols"

os.makedirs(VIZ_SAVE_DIR, exist_ok=True)


# Build examples_df

examples_df = val_df.sample(N_SAMPLES, random_state=42).reset_index(drop=True)

# Ensure label_name exists
if "label_name" not in examples_df.columns:
    if "labels" in examples_df.columns:
        examples_df["label_name"] = examples_df["labels"].apply(
            lambda xs: xs[0] if isinstance(xs, list) and len(xs) else "No Finding"
        )
    else:
        examples_df["label_name"] = examples_df[LBL_COL].astype(str).str.split("|").str[0].str.strip()
        examples_df.loc[examples_df["label_name"].str.lower() == "no finding", "label_name"] = "No Finding"

# Ensure prompt exists
if "prompt" not in examples_df.columns:
    examples_df["prompt"] = examples_df[LBL_COL].apply(labels_to_prompt)

# Output paths 
examples_df["base_path"] = [os.path.join(VIZ_SAVE_DIR, f"base_{i:02d}.png") for i in range(len(examples_df))]
examples_df["lora_path"] = [os.path.join(VIZ_SAVE_DIR, f"lora_{i:02d}.png") for i in range(len(examples_df))]

examples_df["base_text"] = [""] * len(examples_df)
examples_df["lora_text"] = [""] * len(examples_df)

print("Examples preview:")
print(examples_df[["label_name", "prompt"]].head(3))


# Generate images (Base + LoRA) and SAVE AS GRAYSCALE

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

base_pipe.to(device)
lora_pipe.to(device)

try:
    base_pipe.set_progress_bar_config(disable=True)
    lora_pipe.set_progress_bar_config(disable=True)
except Exception:
    pass

use_amp = (device == "cuda")

class _DummyCtx:
    def __enter__(self): return None
    def __exit__(self, *args): return False

for i, row in examples_df.iterrows():
    prompt = row["prompt"]
    seed = BASE_SEED + i
    print(f"[{i}] seed={seed} label={row['label_name']} | prompt[:90]={prompt[:90]}")

    with torch.inference_mode():
        autocast_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else _DummyCtx()
        with autocast_ctx:
            # Base
            g = torch.Generator(device=device).manual_seed(seed)
            base_img = base_pipe(
                prompt=prompt,
                negative_prompt=NEG_PROMPT,
                guidance_scale=GUIDANCE,
                num_inference_steps=STEPS,
                generator=g,
                height=IMG_SIZE,
                width=IMG_SIZE,
            ).images[0]

            # LoRA 
            g = torch.Generator(device=device).manual_seed(seed)
            lora_img = lora_pipe(
                prompt=prompt,
                negative_prompt=NEG_PROMPT,
                guidance_scale=GUIDANCE,
                num_inference_steps=STEPS,
                generator=g,
                height=IMG_SIZE,
                width=IMG_SIZE,
            ).images[0]

    # Convert outputs to grayscale before saving
    base_img.convert("L").save(row["base_path"])
    lora_img.convert("L").save(row["lora_path"])

print("Saved grayscale base/lora images to:", VIZ_SAVE_DIR)


# Build GRAYSCALE comparison sheet 
out_sheet = make_comparison_sheet_gray(
    examples_df,
    out_path=SHEET_PATH,
    label_col="label_name",
    prompt_col="prompt",
    base_img_col="base_path",
    lora_img_col="lora_path",
    base_txt_col="base_text",
    lora_txt_col="lora_text",
    n=N_SAMPLES,
    img_size=256,
    pad=18
)

print("Saved grayscale sheet:", out_sheet)


In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
import torch
import torchxrayvision as xrv

# Choose eval_df (ALL samples)

eval_df = val_sel.copy().reset_index(drop=True)   # run on all selected validation samples
print("eval_df size:", len(eval_df))


# Device

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


# Output / caching

EVAL_CACHE_DIR = os.path.join(OUT_DIR, "label_fidelity_cache_imgs")
os.makedirs(EVAL_CACHE_DIR, exist_ok=True)

SAVE_CSV_PATH = os.path.join(OUT_DIR, "label_fidelity_all.csv")
SAVE_EVERY = 25   


EVAL_SEEDS = CFG.get("eval_seeds", [0, 1])                  
EVAL_STEPS = int(CFG.get("num_inference_steps", 30))       
EVAL_GUIDANCE = float(CFG.get("guidance_scale", 7.5))       

NEG_PROMPT = "normal lungs, no abnormality, clear lungs, perfectly normal, text, letters, watermark, symbols"


# Load XRV model

cxr_model = xrv.models.DenseNet(weights="densenet121-res224-all").to(device).eval()
xrv_pathologies = list(cxr_model.pathologies)

your_classes = [
    "Infiltration","Effusion","Atelectasis","Nodule","Mass","Consolidation","Pleural_Thickening",
    "Pneumothorax","Cardiomegaly","Emphysema","Edema","Fibrosis","Pneumonia","No Finding","Hernia"
]
print("XRV pathologies intersection:",
      sorted(set([c.replace("_"," ") for c in your_classes]).intersection(set(xrv_pathologies))))


# Helpers

def preprocess_for_xrv(pil_img: Image.Image) -> torch.Tensor:
    img = pil_img.convert("L").resize((224, 224), resample=Image.BILINEAR)
    arr = np.array(img).astype(np.float32)
    arr = xrv.datasets.normalize(arr, maxval=255)
    return torch.from_numpy(arr).unsqueeze(0)  # [1,224,224]

def parse_nih_labels(label_str: str):
    s = str(label_str).strip()
    if s.lower() == "no finding":
        return ["No Finding"]
    return [x.strip() for x in s.split("|") if x.strip()]

def map_label_to_xrv(lbl: str):
    lbl = lbl.replace("_", " ").strip()
    if lbl.lower() == "no finding":
        return None
    return lbl

@torch.no_grad()
def gen_one_cached(pipe, prompt: str, seed: int, cache_path: str):
    """
    Generates one image and caches it at cache_path.
    If cache exists, loads it instead.
    """
    if os.path.exists(cache_path):
        return Image.open(cache_path).convert("RGB")

    g = torch.Generator(device=device).manual_seed(int(seed))
    out = pipe(
        prompt=prompt,
        negative_prompt=NEG_PROMPT,
        guidance_scale=EVAL_GUIDANCE,
        num_inference_steps=EVAL_STEPS,
        generator=g,
    )
    img = out.images[0]
    img.save(cache_path)
    return img

@torch.no_grad()
def target_prob(pipe, label_str, prompt, seeds, cache_prefix: str):
    targets = [map_label_to_xrv(x) for x in parse_nih_labels(label_str)]
    targets = [t for t in targets if t is not None and t in xrv_pathologies]
    if len(targets) == 0:
        return np.nan, ""

    imgs = []
    for s in seeds:
        cache_path = os.path.join(EVAL_CACHE_DIR, f"{cache_prefix}_seed{s}.png")
        imgs.append(gen_one_cached(pipe, prompt, s, cache_path))

    batch = torch.stack([preprocess_for_xrv(im) for im in imgs], dim=0).to(device)  
    logits = cxr_model(batch)
    probs = torch.sigmoid(logits).detach().cpu().numpy()

    idxs = [xrv_pathologies.index(t) for t in targets]
    return float(probs[:, idxs].mean()), ", ".join(targets)


done_keys = set()
records = []

if os.path.exists(SAVE_CSV_PATH):
    prev = pd.read_csv(SAVE_CSV_PATH)
    # unique key per row (index-based in eval_df)
    if "eval_index" in prev.columns:
        done_keys = set(prev["eval_index"].astype(int).tolist())
        records = prev.to_dict("records")
    print(f"Resuming: loaded {len(done_keys)} completed rows from {SAVE_CSV_PATH}")


# Run label fidelity on ALL rows

eval_labels = eval_df[LBL_COL].tolist()
eval_prompts = eval_df["prompt"].tolist()

for idx, (label_str, prompt) in enumerate(tqdm(list(zip(eval_labels, eval_prompts)), desc="label fidelity (all)")):
    if idx in done_keys:
        continue

    # cache prefix includes which pipe, index
    base_prefix = f"base_idx{idx:05d}"
    lora_prefix = f"lora_idx{idx:05d}"

    b, used = target_prob(base_pipe, label_str, prompt, EVAL_SEEDS, cache_prefix=base_prefix)
    l, _    = target_prob(lora_pipe, label_str, prompt, EVAL_SEEDS, cache_prefix=lora_prefix)

    rec = {"eval_index": idx,
        "label_str": label_str,
        "targets_used": used,
        "base_target_mean_prob": b,
        "lora_target_mean_prob": l,
        "delta_lora_minus_base": (l - b) if (np.isfinite(l) and np.isfinite(b)) else np.nan
    }
    records.append(rec)
    done_keys.add(idx)

    if (len(done_keys) % SAVE_EVERY) == 0:
        pd.DataFrame(records).to_csv(SAVE_CSV_PATH, index=False)
        print(f"Saved progress: {len(done_keys)}/{len(eval_df)} -> {SAVE_CSV_PATH}")

# final save
lf_df = pd.DataFrame(records).sort_values("eval_index").reset_index(drop=True)
lf_df.to_csv(SAVE_CSV_PATH, index=False)

display(lf_df.head(25))
print("Saved:", SAVE_CSV_PATH)
print("Mean delta (ignoring NaNs):", lf_df["delta_lora_minus_base"].mean(skipna=True))
print("Non-NaN rows:", lf_df["delta_lora_minus_base"].notna().sum(), "/", len(lf_df))


In [ ]:
if CFG["run_xrv_eval"]:
    cxr_model = xrv.models.DenseNet(weights="densenet121-res224-all").to(device).eval()
    xrv_pathologies = list(cxr_model.pathologies)

    # NIH labels do not perfectly match xrv; we filter those that exist
    def parse_nih_labels(label_str: str):
        s = str(label_str).strip()
        if s.lower() == "no finding":
            return ["No Finding"]
        return [x.strip() for x in s.split("|") if x.strip()]

    @torch.no_grad()
    def preprocess_for_xrv(pil_img: Image.Image) -> torch.Tensor:
        img = pil_img.convert("L").resize((224,224), resample=Image.BILINEAR)
        arr = np.array(img).astype(np.float32)
        arr = xrv.datasets.normalize(arr, maxval=255)
        return torch.from_numpy(arr).unsqueeze(0)  # 1x224x224

    def keep_targets(label_str):
        labs = parse_nih_labels(label_str)
        labs = [l for l in labs if l != "No Finding" and l in xrv_pathologies]
        return labs

    @torch.no_grad()
    def target_prob(pipe_, label_str, prompt, seeds):
        targets = keep_targets(label_str)
        if len(targets) == 0:
            return np.nan, ""

        imgs = [gen_one(pipe_, prompt, s) for s in seeds]
        batch = torch.stack([preprocess_for_xrv(im) for im in imgs], dim=0).to(device, non_blocking=True)

        probs = torch.sigmoid(cxr_model(batch)).detach().cpu().numpy()
        idxs = [xrv_pathologies.index(t) for t in targets]
        return float(probs[:, idxs].mean()), ", ".join(targets)

    records = []
    for i, (label_str, prompt) in enumerate(tqdm(list(zip(eval_labels, eval_prompts)), desc="label fidelity")):
        b, used = target_prob(base_pipe, label_str, prompt, CFG["eval_seeds"])
        l, _    = target_prob(lora_pipe, label_str, prompt, CFG["eval_seeds"])
        records.append({
            "prompt_id": i,
            "label_str": label_str,
            "targets_used": used,
            "base_target_mean_prob": b,
            "lora_target_mean_prob": l,
            "delta_lora_minus_base": (l - b) if (np.isfinite(l) and np.isfinite(b)) else np.nan
        })

    lf_df = pd.DataFrame(records)
    lf_csv = os.path.join(OUT_DIR, "label_fidelity.csv")
    lf_df.to_csv(lf_csv, index=False)
    display(lf_df.head())
    print("Saved:", lf_csv)

    valid = lf_df.dropna(subset=["delta_lora_minus_base"]).copy()
    valid["improved"] = valid["delta_lora_minus_base"] > 0
    summary = {
        "n_prompts": int(len(valid)),
        "mean_delta": float(valid["delta_lora_minus_base"].mean()) if len(valid) else float("nan"),
        "median_delta": float(valid["delta_lora_minus_base"].median()) if len(valid) else float("nan"),
        "pct_improved": float(valid["improved"].mean() * 100) if len(valid) else float("nan"),
    }
    with open(os.path.join(OUT_DIR, "label_fidelity_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("LABEL FIDELITY SUMMARY:", summary)

    # plot
    if len(valid):
        plot_df = valid.sort_values("delta_lora_minus_base")
        plt.figure(figsize=(8,6))
        plt.barh(plot_df["targets_used"], plot_df["delta_lora_minus_base"])
        plt.xlabel("Δ target probability (LoRA − Base)")
        plt.title("Pathology alignment improvement via LoRA fine-tuning")
        plt.tight_layout()
        plot_path = os.path.join(OUT_DIR, "label_fidelity_delta_barplot.png")
        plt.savefig(plot_path, dpi=200)
        plt.show()
        print("Saved:", plot_path)


In [ ]:
if CFG["run_lpips"]:
    loss_fn = lpips.LPIPS(net="alex").to(device).eval()

    def img_to_tensor(im):
        arr = np.array(im).astype(np.float32) / 255.0
        t = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(device) * 2 - 1
        return t

    lpips_rows = []
    for i, p in enumerate(tqdm(eval_prompts, desc="LPIPS")):
        imgs = [gen_one(lora_pipe, p, s) for s in CFG["eval_seeds"]]
        ts = [img_to_tensor(x) for x in imgs]
        dists = []
        for a in range(len(ts)):
            for b in range(a+1, len(ts)):
                dists.append(float(loss_fn(ts[a], ts[b]).item()))
        lpips_rows.append({"prompt_id": i, "lpips_mean": float(np.mean(dists))})

    lpips_df = pd.DataFrame(lpips_rows)
    lpips_df.to_csv(os.path.join(OUT_DIR, "lpips.csv"), index=False)

    summary = {"lpips_mean_over_prompts": float(lpips_df["lpips_mean"].mean())}
    with open(os.path.join(OUT_DIR, "lpips_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print("LPIPS summary:", summary)
